# Week 6 Clustering Practice (Solution)

This notebook contains a complete worked version of the student activity. It is intentionally separated from the student notebook so the learning exercise stays focused on reasoning and interpretation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")

base, _ = make_blobs(
    n_samples=300,
    centers=[(-4, -5, 0, 0), (0, 0, 0, 0), (5, 5, 8, 8)],
    cluster_std=[1.0, 1.4, 1.1],
    random_state=42,
)
X_raw = np.column_stack([base[:, 0], base[:, 1] * 25, base[:, 2], base[:, 3] * 0.5])
feature_names = ["feature_1", "feature_2", "feature_3", "feature_4"]
df = pd.DataFrame(X_raw, columns=feature_names)
X_scaled = StandardScaler().fit_transform(X_raw)

raw_model = KMeans(n_clusters=3, n_init=20, random_state=42)
scaled_model = KMeans(n_clusters=3, n_init=20, random_state=42)
raw_labels = raw_model.fit_predict(X_raw)
scaled_labels = scaled_model.fit_predict(X_scaled)

print(f"Raw silhouette score: {silhouette_score(X_raw, raw_labels):.3f}")
print(f"Scaled silhouette score: {silhouette_score(X_scaled, scaled_labels):.3f}")
print(pd.Series(raw_labels).value_counts().sort_index().to_dict())
print(pd.Series(scaled_labels).value_counts().sort_index().to_dict())


## What the scaling comparison shows

The same data are being clustered under two different representations. Because `feature_2` is multiplied by 25, it dominates Euclidean distance when no scaling is applied. The scaled version makes all features contribute more evenly to the distance calculations, which changes the assignments and usually raises the silhouette score.

This is the core modeling lesson: a cluster is not a fact about the data alone; it is a fact about the data representation and the chosen feature scale.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
sns.scatterplot(data=df.assign(cluster=raw_labels), x="feature_1", y="feature_2", hue="cluster", palette="Set2", s=40, ax=axes[0])
axes[0].set_title("Clusters on raw features")
scaled_df = df.copy()
scaled_df[["feature_1", "feature_2", "feature_3", "feature_4"]] = X_scaled
sns.scatterplot(data=scaled_df.assign(cluster=scaled_labels), x="feature_1", y="feature_2", hue="cluster", palette="Set2", s=40, ax=axes[1])
axes[1].set_title("Clusters after scaling")
plt.tight_layout()
plt.show()


In [ ]:
ks = range(2, 7)
inertia = []
silhouettes = []
for k in ks:
    model = KMeans(n_clusters=k, n_init=25, random_state=42)
    labels = model.fit_predict(X_scaled)
    inertia.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inertia, marker='o')
axes[0].set_title("Elbow plot (WSS / inertia)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(ks), silhouettes, marker='o', color='C1')
axes[1].set_title("Silhouette score by K")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()
for k, sil in zip(ks, silhouettes):
    print(f"K = {k}: silhouette = {sil:.3f}")


## Reasonable K choice

For this synthetic dataset, a reasonable choice is K = 3. The inertia drops sharply from K = 2 to K = 3 and then flattens, while the silhouette score is strongest at K = 3. We still do not treat these results as a proof of the true number of clusters; instead, we use them as evidence alongside interpretability and stability.

If K = 4 had a similar silhouette, we would inspect whether the extra cluster was splitting a meaningful group or creating a fragile partition.


In [ ]:
k_choice = 3
final_model = KMeans(n_clusters=k_choice, n_init=25, random_state=42)
df["cluster"] = final_model.fit_predict(X_scaled)
cluster_profile = df.groupby("cluster")[feature_names].mean().round(2)
cluster_profile


## Cluster interpretation

This dataset was generated with three latent groups, so the cluster means naturally separate into low, middle, and high feature values. The cluster labels (0, 1, 2) are not explanations by themselves; we interpret them by reading the feature means and asking what each group looks like in the original variables.

A useful interpretation is therefore not 'cluster 1 is a natural category.' It is: 'cluster 1 is the group with the highest values on feature_2 and feature_4, cluster 0 is the group with lower values on several features, and cluster 2 sits between the extremes.'

This matters because cluster labels are algorithmic outputs, not discovered reality.


In [ ]:
alt_model = KMeans(n_clusters=4, n_init=25, random_state=42)
alt_labels = alt_model.fit_predict(X_scaled)
print(f"Silhouette for 4 clusters: {silhouette_score(X_scaled, alt_labels):.3f}")
print(pd.Series(alt_labels).value_counts().sort_index().to_dict())


## Sensitivity and stability

Changing one reasonable modeling choice—here, moving from K = 3 to K = 4—can change the cluster assignments. The substantive interpretation stays broadly similar if the new clusters are just splitting a group rather than changing the overall pattern. If the interpretation changes dramatically, that is evidence that the clusters are not robust enough to use as a strong conclusion.


In [ ]:
Z = linkage(X_scaled, method='ward')
plt.figure(figsize=(10, 5))
dendrogram(Z, p=12, truncate_mode='lastp', no_labels=True)
plt.title('Hierarchical clustering dendrogram')
plt.xlabel('Observation index')
plt.ylabel('Ward distance')
plt.tight_layout()
plt.show()


## Hierarchical clustering companion

The dendrogram shows the nested structure that appears when clusters are formed by merging similar groups. Cutting the dendrogram at a chosen height yields a set of clusters, but that choice still matters. The hierarchical view is a complementary way to reason about structure; it is not a separate truth about the one correct grouping.


## AI critique

The claim "The elbow plot proves the dataset has three natural groups" is too strong. An elbow plot gives evidence about a reasonable K, but it does not reveal the true structure in the data. We would want to inspect the silhouette score, cluster profiles, and the stability of the interpretation under reasonable modeling changes before making a stronger claim.
